In [1]:
from IPython.display import HTML

HTML(
    '<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">'
)

# <i class="fas fa-rocket" style="color:#18ab4b"></i> Endpoint deployment: real-time credit scoring

Train the CatBoost model with `ModelTrainer`, then serve it as a <i class="fas fa-rocket" style="color:#18ab4b"></i> **local SageMaker endpoint**: the same `create_model` / `create_endpoint_config` / `create_endpoint` calls as on AWS, made through the SDK v3 `LocalSession`, which runs the serving container (`<image> serve`) on `:8080`. The endpoint scores loan applications and returns <i class="fas fa-play-circle" style="color:#18ab4b"></i> probability of default and a SHAP-based scorecard score. Mirrors `sagemaker_endpoints/.../sagemaker_endpoint.py` + `inference.py`.

Prereqs: Docker, `pip install 'oblako[sagemaker,notebook]'`.

In [2]:
import logging

# the SageMaker SDK logs every step at INFO through a root rich handler, unless the
# root logger is configured first: keep its output (and botocore's) to warnings
logging.basicConfig(level=logging.WARNING)
logging.getLogger("sagemaker.config").setLevel(logging.WARNING)
# a local endpoint is polled on /ping while it starts; the retries aren't news
logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)

## <i class="fas fa-user-graduate" style="color:#18ab4b"></i> Build + train

In [3]:
import os
import shutil
from pathlib import Path

import creditdata
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, InputData
from sagemaker.train.model_trainer import Mode

from oblako.engines.sagemaker import use_local_stubs
from oblako.services import SageMakerService

os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
use_local_stubs()  # no AWS account: skip the SDK's IAM role check

IMAGE = "oblako-credit-risk:latest"
HP = {
    "iterations": "150",
    "depth": "6",
    "learning-rate": "0.1",
    "factor": "20",
    "offset": "600",
}

sm = SageMakerService()
sm.build_image(path="container", tag=IMAGE)  # CatBoost BYOC (./container)

# ModelTrainer's local mode bind-mounts its working dir into the container, so keep
# it under the project: Docker Desktop shares /Users, not the /var/folders temp dir.
work = Path(".sm-local/02").resolve()
shutil.rmtree(work, ignore_errors=True)
(work / "train").mkdir(parents=True)
(work / "job").mkdir()
creditdata.write_training_csv(str(work / "train" / "train.csv"), n=600)

trainer = ModelTrainer(
    training_image=IMAGE,
    training_mode=Mode.LOCAL_CONTAINER,
    compute=Compute(instance_type="local", instance_count=1),
    role="arn:aws:iam::000000000000:role/dummy",
    hyperparameters=HP,
    local_container_root=str(work / "job"),
)
trainer.train(
    input_data_config=[InputData(channel_name="train", data_source=str(work / "train"))]
)
model_tar = work / "job" / "compressed_artifacts" / "model.tar.gz"
print("model:", model_tar.relative_to(Path.cwd()))

Step 1/7 : FROM python:3.11-slim
 ---> 1042b61448fe
Step 2/7 : RUN apt-get update && apt-get install -y --no-install-recommends libgomp1     && rm -rf /var/lib/apt/lists/*
 ---> Using cache
 ---> 6f2747b28505
Step 3/7 : RUN pip install --no-cache-dir catboost numpy pandas joblib
 ---> Using cache
 ---> b722ab224512
Step 4/7 : COPY src/ /opt/program/
 ---> Using cache
 ---> e61966507729
Step 5/7 : WORKDIR /opt/program
 ---> Using cache
 ---> 260c052be1a8
Step 6/7 : ENV PYTHONUNBUFFERED=1
 ---> Using cache
 ---> 6d875257c67c
Step 7/7 : ENTRYPOINT ["python", "main.py"]
 ---> Using cache
 ---> e3ca3a20dd4e
Successfully built e3ca3a20dd4e
Successfully tagged oblako-credit-risk:latest


time="2026-09-30T18:57:18+02:00" level=warning msg="a network with name sagemaker-local exists but was not created for project \"job\".\nSet `external: true` to use an existing network"
 Container job-algo-1-1 Recreate 
 Container job-algo-1-1 Recreated 
Attaching to algo-1-1
 Container job-algo-1-1 Starting 


 Container job-algo-1-1 Started 


algo-1-1  | trained CatBoost on 600 rows, 12 features (default rate 0.39)
algo-1-1 exited with code 0
 Compose Stopping Aborting on container exit...
 Container job-algo-1-1 Stopping 
 Container job-algo-1-1 Stopped 


model: .sm-local/02/job/compressed_artifacts/model.tar.gz


## <i class="fas fa-rocket" style="color:#18ab4b"></i> Deploy a local endpoint and <i class="fas fa-play-circle" style="color:#18ab4b"></i> score applications

`LocalSession` is used directly rather than `ModelBuilder.deploy_local()`: in SDK 3.17 `deploy_local` starts the serving container and then registers the endpoint through `LocalSession.create_endpoint`, which starts a second container on the same port and blocks.

In [4]:
import importlib
import json

from sagemaker.core.local import LocalSession

# the local serving path imports sagemaker.serve.model_builder lazily; load it up front
importlib.import_module("sagemaker.serve.model_builder")

NAME = "credit-risk"
session = LocalSession()
session.config = {"local": {"local_code": True}}
sm_client = session.sagemaker_client
sm_client.create_model(
    ModelName=NAME,
    PrimaryContainer={
        "Image": IMAGE,
        "ModelDataUrl": f"file://{model_tar}",
        "Environment": {},  # local mode reads it unconditionally
    },
)
sm_client.create_endpoint_config(
    EndpointConfigName=NAME,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": NAME,
            "InitialInstanceCount": 1,
            "InstanceType": "local",
        }
    ],
)
sm_client.create_endpoint(EndpointName=NAME, EndpointConfigName=NAME)
print("endpoint:", sm_client.describe_endpoint(EndpointName=NAME)["EndpointStatus"])

apps = creditdata.sample_applications()
resp = session.sagemaker_runtime_client.invoke_endpoint(
    EndpointName=NAME,
    ContentType="application/json",
    Body=json.dumps({"instances": apps}),
)
result = json.loads(resp["Body"].read())
for app, pred in zip(apps, result["predictions"]):
    print(
        f"app_score={app['Application_Score']} bureau={app['Bureau_Score']} "
        f"l2i={app['Loan_to_income']:.0f} {app['Residential_Status']:<20} "
        f"-> PD={pred['proba']:.3f} score={pred['score']}"
    )

Attaching to v7uxdfush3-algo-1-5jqtl


endpoint: InService
app_score=820 bureau=800 l2i=8 Owner                -> PD=0.025 score=673
app_score=420 bureau=450 l2i=150 Tenant               -> PD=0.974 score=527


In [5]:
sm_client.delete_endpoint(EndpointName=NAME)
sm_client.delete_endpoint_config(EndpointConfigName=NAME)
sm_client.delete_model(ModelName=NAME)
print("endpoint torn down")

endpoint torn down
